# darts_forecast intraday: which bar size, input span and output span predict the rest of the day best?

**In one sentence.** For a small, diverse set of stocks, a Darts model reads the last minutes of trading
(today's, and if the input reaches back, yesterday's last ones) and predicts the path of the price for the
next minutes **of the same day**, never past the 16:00 close; every combination of **bar size** (1, 2, 3, 5,
10 minutes), **input span** (30 minutes to a whole session) and **output span** (10 minutes to the rest of
the day) is measured, and the paths are graded point by point against what really happened.

This is the intraday sequel of `darts_forecast_explorer.ipynb`, whose paths ran through the night: here the
night, the weekend and the holidays are **never inside a predicted path or a target**. The overnight gap is
known at the open, so it is an input. Like the explorer it is a **workbench for a baseline**: it does not
trade.

**Where each part of the request is answered**

| you asked for | where |
|---|---|
| intraday only, a hard stop at the end of the trading day | §3 (one series per stock-day; a path never passes 16:00) |
| the night, weekends and holidays handled separately | §3 (the gap is an input known at the open, never part of a target); §4 (a check that moves every night at random and finds every target unchanged) |
| bar sizes 1, 2, 3, 5 and 10 minutes, so there are enough data points | §0, §8; §8 also shows what finer bars do and do not add |
| input data from the day before is fine, but the model must know when each input bar is from | §3: a "today" flag (0 on yesterday's bars, 1 on today's) and the time of day on every bar, plus the calendar days since the previous session |
| "always use the first x minutes of the day to predict the rest of the day?" | §10 (model-free: which part of the day predicts which), and the 10:00 → close forecast in §9 |
| the agreement of the whole predicted path, averaged over all stocks, one model for all | §5 (the grade), §8 |
| how the knobs change the picture | §9 |

**Roadmap**

| § | question |
|---|---|
| 0 | which settings, what can go wrong, and what do the words mean? |
| 1 | which stocks? |
| 2 | is their data complete, what is the shape of a trading day, and what is a move worth? |
| 3 | what exactly is predicted, and how does the model know when each input bar is from? |
| 4 | which months train, which are graded, and how do we know nothing leaks? |
| 5 | one configuration end to end |
| 6 | which Darts model? |
| 7 | could this pipeline find an effect if there were one? (a planted test) |
| 8 | the grid: bar size × input span × output span, every cell measured |
| 9 | how do the three knobs change the picture? |
| 10 | does the start of the day predict the rest of it? |
| 11 | does the winner hold on a month nobody looked at? |
| 12 | the checks, the summary, and what to try next |

**Run time.** `SMOKE = True` runs every step small in about two minutes. The full run takes about an hour
the first time (the grid is most of it: 150 configurations, 5 to 40 seconds each); every result is saved in
`cache/`, so a re-run takes about a minute.

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel, or run  uv sync --all-extras
import importlib.util
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger("darts").setLevel(logging.ERROR)  # Darts' notes about optional models it lacks

import pandas as pd

from trading_simple import strategies
from trading_simple.config import load_settings
from trading_simple.db import make_engine


def helpers(name: str):
    "One of the strategy folder's helper modules, loaded by path (strategy.py imports none)."
    path = strategies.STRATEGIES_DIR / "darts_forecast" / f"{name}.py"
    spec = importlib.util.spec_from_file_location(f"darts_{name}", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


ix, charts = helpers("intraday"), helpers("charts")
ex, rs = ix.ex, ix.rs  # explorer.py and research.py, as loaded by the intraday module
settings = load_settings()
engine = make_engine(settings)
pd.set_option("display.max_colwidth", 90)

## 0 · The settings

**Question: which settings, what can go wrong, and what do the words mean?**

Every setting is in the next cell. `SMOKE = False` (the default) is the real run. Switch on `SMOKE = True`
to test every step end to end in about two minutes: 4 stocks, 3 months, a 2 × 2 × 2 grid, one training
pass. Use it after any code change.

In [ ]:
SMOKE = False  # True: every step, small and fast (a test); False: the real run
MODEL = "tide"  # the Darts model of the grid: tide | tsmixer | dlinear (§6 compares them)
WINDOW = ("2025-08-01", "2026-01-31")  # the months studied (inside TUNE, see below)
TRAIN_MONTHS, TEST_MONTHS = 2, 1  # the first fit learns from 2 months; the last month is TEST
GRANULARITIES = list(ix.GRANULARITIES)  # bar sizes measured (minutes)
INPUT_SPANS = list(ix.INPUT_SPANS)  # minutes the model reads (390 = a whole session)
OUTPUT_SPANS = list(ix.OUTPUT_SPANS)  # minutes it predicts (360 = from 10:00 to the close)
EPOCHS = 10  # at most this many training passes (stopped early on a held-out week)
DEFAULT = (5, 120, 60)  # (bar minutes, input minutes, output minutes) of §5, §6 and §7
COMPARE = ["tide", "tsmixer", "dlinear"]  # the models of §6
PLANTED = list(ix.PLANTED)  # §7's planted effects: correlation with the 30-minute move
DRAWS = 200  # make-believe futures behind every luck band
ROUND_TRIP_BP = 10  # what a round trip costs (bp): commission + half the spread, twice
SYMBOLS = None  # None: the picks of §1; or your own list, e.g. ["AAPL", "KO", "SMCI"]
READ_LOCKED = False  # True allows a WINDOW after the TUNE months (see below)
READ_TEST = True  # §11 reads the TEST month once; False keeps it unread

K = ix.knobs(smoke=SMOKE, model=MODEL, window=WINDOW, train_months=TRAIN_MONTHS,
             test_months=TEST_MONTHS, granularities=GRANULARITIES, inputs=INPUT_SPANS,
             outputs=OUTPUT_SPANS, epochs=EPOCHS, draws=DRAWS, compare=COMPARE,
             default=DEFAULT, planted=PLANTED, round_trip_bp=ROUND_TRIP_BP)
CHECKS = []  # every section adds its checks; §12 shows them all
display(ix.knob_table(K))

**Why these settings**

| setting | value | why |
|---|---|---|
| `GRANULARITIES` | 1, 2, 3, 5, 10 | your list. Every bar is built from the stored 1-minute bars; each divides the 390-minute session |
| `INPUT_SPANS` | 30, 60, 120, 240, 390 minutes | in **minutes**, not bars: "the last hour" is the same question at every bar size (60 bars of 1 minute, 6 of 10). 390 = one whole session, so the input reaches at most into yesterday, as you asked |
| `OUTPUT_SPANS` | 10, 30, 60, 120, 240, 360 minutes | in minutes for the same reason; a path never passes 16:00, so a 360-minute path starts only at 10:00 (the rest of the day) and a 10-minute one at every half hour |
| `MODEL` | `tide` | TiDE predicts the whole path at once and reads the bar inputs, the clock and the day's facts; §6 compares TSMixer and DLinear |
| `EPOCHS` | 10 | at most: training stops when the error on a held-out week stops improving for 2 passes, and keeps the best pass |
| `WINDOW` | Aug 2025 → Jan 2026 | six months inside the darts study's TUNE months: the locked TEST (Feb–Apr 2026) and HOLDOUT (May 2026 on) stay unread. `READ_LOCKED = True` allows a later window |
| `DEFAULT` | 5-min bars, input 120 min, predict 60 min | the worked example: "from the last two hours of 5-minute bars, predict the next hour, bar by bar" |
| `ROUND_TRIP_BP` | 10 | about 1 bp of commission and 4 bp of half-spread per side for these stocks: it only draws §2's break-even line |

Lengths in minutes replace the explorer's lengths in bars: with a hard stop at the close, 100 bars cannot be
predicted on 10-minute bars (a session has 39), and "100 bars ahead" would mean 100 minutes at one bar size
and 1,000 at another.

**What can go wrong** (each point was measured on these stocks; the notebook guards against each)

| trap | what it does | the guard |
|---|---|---|
| the night leaks into a target | a session's first move would hold the gap: the model "predicts" news it saw at the open | every bar's move is measured inside its own session (the first from the open); a check moves every night at random and requires every target to stay the same (§4) |
| the decision bar's last trade is not a price you can trade at | on 1-minute bars a move tends to reverse (bid-ask bounce, stale prints): free, untradable "skill" | every graded path starts at the **fill**, the first trade after the decision; volume-0 placeholder minutes (a copy of the last price, 0–8% of minutes here) are masked (§2) |
| each bar size graded on its own yardstick | averaging R² over a grid's own lead times favours whichever grid has more of the easy ones | one truth for every configuration: the move from the fill to fixed checkpoints (10, 20, 30, 60, … 360 minutes), in one typical-move unit (§5) |
| a network's paths are too loud | for stock returns the best squared-error forecast is close to "no change" | each configuration's paths are scaled by β, fit month by month on earlier months only (§5) |
| finer bars look like more data | without a limit, 1-minute bars would give 10× the training windows of 10-minute bars, all overlapping: the same **independent** paths | every bar size trains on one window about every 10 minutes of the clock, so the same windows; finer bars only add detail inside each input (§8) |
| baselines that lose by construction | a time-of-day mean over 20 noisy days scores below 0 unscaled (−0.01 to −0.02 here) | both baselines (the time-of-day mean path, a ridge on a dozen summaries) get their own month-by-month scale, fit the way the model's is (§5) |
| a "nothing found" that means nothing | 12 stocks and 3 months can hide a small effect | §7 plants a known effect and checks the pipeline finds it; every row says the smallest R² its data could have detected |
| the best of many tries flatters itself | with 150 tries, some look good by luck | a luck band, the bar for the best of all tries, and one pre-registered TEST read (§9, §11) |
| a luck band that shuffles days | a day's half-hours share its volatility; shuffling days breaks that and narrows the band: in pure noise 10–22% of the map's cells passed 1.96 | every luck band flips signs and keeps every move's size: per 10-minute block of the clock for the paths, per day for the map (§5, §10) |

**Words you will meet**

| word | meaning |
|---|---|
| bar, granularity | *g* minutes of trading built from the 1-minute bars; a session (09:30–16:00 New York) has 390 / *g* |
| stock-day series | what the model learns from and predicts on: one stock, one session, plus the previous session's last minutes as input |
| input span, output span | the minutes the model reads (ending at the decision), the minutes it predicts (bar by bar, all at once) |
| decision | the moment a path is predicted: 10:00, 10:30, … 15:30 (every bar size closes a bar there) |
| fill | the first trade after the decision: where a graded path starts |
| checkpoint | a graded point of the path: 10, 20, 30, 60, 90, 120, 180, 240, 300 and 360 minutes after the decision (each configuration up to its output span) |
| typical move | the usual size of a move for that stock, that time of day and that length, from earlier sessions only; every target and every graded move is in these units |
| R² against "no change" | 1 − (squared error of the predicted path) / (squared error of predicting no move), per stock and checkpoint: 0 = as good as saying nothing, above 0 = better. **The grade**, averaged over checkpoints and stocks |
| scale (β) | how much of a model's path proved right on earlier data: 0 = none (the path becomes "no change"), 1 = all |
| direction (corr), luck band, z | the correlation of predicted and actual change; where it lands when the outcomes have no direction (every 10-minute block of every future gets a random sign); z = how many luck-SDs above luck's centre (0) |
| detectable R² | the smallest R² this many graded paths would show 80% of the time (for one pre-registered test, and for the best of all the tries) |

## 1 · The stocks: small, diverse, with complete 1-minute data

**Question: which stocks, and why these?**

The same rules and picks as the explorer (`darts_forecast_explorer.ipynb` §1): every stock with complete
1-minute data in the window and over the last six months, a trade in at least 90% of its minutes, a US
listing with a market cap and a price of at least $5. Two stocks per cap group (large ≥ $10 B, mid $2–10 B,
small < $2 B, today's caps) and speed (the daily volatility before the window, nearest the 10th or 90th
percentile of its group): 12 stocks. The rules look at data quality and size, never at a return; the one
mild bias is that a stock must still be listed today.

In [ ]:
cands = ex.candidates(engine, WINDOW)  # every stock with 1-minute data in the window
print(f"window {WINDOW[0]} → {WINDOW[1]}: {cands.attrs['sessions']} sessions; "
      f"market caps: {cands.attrs['listing']}")
display(ex.rule_funnel(cands))
picks = ex.pick(cands)  # two per cap group and speed, nearest the target percentile, sectors spread
display(ex.pick_table(picks))
charts.universe_map(cands, picks).show()
NAMES = SYMBOLS or ex.default_names(picks, smoke=K["smoke"])  # SMOKE: one of each extreme group
print(f"{len(NAMES)} stocks: {', '.join(NAMES)}")
CHECKS += [("every pick passes the five data rules", bool(cands.loc[picks["symbol"], "ok"].all()),
            f"{len(picks)} picks")]

## 2 · The data: complete, the shape of a day, and what a move is worth

**Question: is the data complete, how does a trading day look, and how big is a move worth predicting?**

The panel holds every regular-session minute of the window plus the 21 sessions before it that the inputs
need for their history (no model learns from them); prices are adjusted for dividends. **Placeholder
minutes**: IBKR stores a minute in which almost nothing traded as a copy of the last price with volume 0.
Left in, they are stale prices that the next trade predictably corrects, so they are masked as "no trade"
before any bar is built.

In [ ]:
panel, SP = ex.load_window(settings, engine, NAMES, K["window"], train_months=K["train_months"],
                           test_months=K["test_months"], read_locked=READ_LOCKED)
dq = ex.data_quality(panel, picks).join(ix.placeholder_table(panel))
print(f"{len(panel.symbols)} stocks, {len(SP['sessions'])} sessions "
      f"{SP['sessions'][0]} → {SP['sessions'][-1]}, {len(panel.close):,} one-minute rows")
display(dq.round(1))
CHECKS += [
    ("every stock has every session", bool((dq["sessions"] == len(SP["sessions"])).all()),
     f"{len(SP['sessions'])} sessions"),
    ("every stock trades in >= 90% of its minutes",
     bool((dq["minutes with a trade %"] >= 90).all()),
     f"lowest {dq['minutes with a trade %'].min():.1f}%"),
    ("no day move above 60% (an unadjusted split would show as one)",
     bool((dq["largest day move %"] < 60).all()), f"largest {dq['largest day move %'].max():.1f}%"),
    ("the window stays inside TUNE (the locked months unread)",
     SP["window"][1] <= rs.WINDOWS["tune"][1] or READ_LOCKED,
     f"window ends {SP['window'][1]}; TUNE ends {rs.WINDOWS['tune'][1]}"),
]
charts.price_paths(panel, picks, SP).show()

**The shape of a day.** The typical 1-minute move at every minute of the session, against the day's
average (all stocks, TRAIN): large just after the open, quiet over lunch, rising into the close. This curve
is the unit of the whole study: every bar's target and every graded move is divided by its typical size at
that time of day, so a calm stock and a wild one, and the open and lunch, count alike.

In [ ]:
charts.day_shape(ix.profile(panel, SP)).show()

**What 1-minute bars hide.** How much a bar's move tends to repeat (positive) or reverse (negative) the
previous bar's within a session (TRAIN). On 1- and 2-minute bars moves tend to reverse: the bid-ask bounce
and stale prints. That is "predictable" but not tradable, which is why every graded path starts at the
fill, not at the decision bar's last trade.

In [ ]:
display(ix.bounce_table(panel, SP, K["granularities"]))

**What a move is worth.** The typical move at each checkpoint for the median stock (TRAIN), and the R² a
forecast would need before trading its direction at every decision pays the round trip (`ROUND_TRIP_BP`).
A short path moves little, so it needs a large R² to pay costs; this is the line to compare every result
with.

In [ ]:
moves = ix.move_table(panel, SP, K["round_trip_bp"])
display(moves)

## 3 · What is predicted, and how the model knows when each input bar is from

**Question: what exactly does the model predict, from what, and how does it tell yesterday from today?**

**One series per stock-day.** Every example the model learns from (and every forecast) is one stock on one
session: the previous session's last minutes (as many as the input span needs, never more than one
session) followed by the whole session to its 16:00 close. Because the input part ends at today's first
bar, every window of the input span holds at least one bar of today, so every bar the model predicts is a
bar of today: **no window reaches past the close, and none crosses a night**. A decision at 15:30 can only
be graded on paths of at most 30 minutes; a 6-hour path only from 10:00.

**The target.** Every bar's intraday move: from the previous bar's close, and for a session's first bar
from its own open, so the night is never inside. It is divided by the bar's typical size (the stock's
typical 1-minute move over the previous 20 sessions × the time-of-day curve of §2). A move beyond 10
typical moves counts as 10 in training (the graded truth is never clipped).

**When each bar is from.** Every bar carries, as inputs known in advance:

- the **time of day** (fraction of the session: 0 at 09:30, and its sine and cosine);
- the **"today" flag**: 0 on the previous session's bars, 1 on today's. The clock alone could not tell
  15:59 yesterday from 09:30 today: on a sine/cosine clock they sit next to each other.

And every stock-day carries, once, the facts of the day known at the open: which stock, the **overnight
gap**, yesterday's move, the last 5 and 20 days, the volatility, the weekday, the **calendar days since the
previous session** (3 after a weekend, 4 after a long one) and the option-expiry week. The night, the
weekend and the holidays reach the model only this way: as information at the open, never as a move to
predict.

**The picture.** One stock-day series at one decision: the grey part is yesterday's end (input only), the
blue band the input span, the green band the predicted span and the diamonds the graded checkpoints; below,
the "today" flag and the time of day the model is given for each bar.

In [ ]:
g0, L0, S0 = K["default"]
NAME, TS = ix.example(panel, picks, NAMES, SP)  # a fast large cap, 10:00, the 5th VALIDATION day
charts.stock_day(ix.day_picture(panel, NAME, TS, K["default"])).show()
display(ix.inputs_table(L0 // g0, g0))

## 4 · The split, and why nothing leaks

**Question: which months train, which are graded, and how do we know no future data leaks into a
prediction?**

| part | what happens |
|---|---|
| **warm-up** (21 sessions) | inputs only: the 20-day inputs and the typical-move unit need history |
| **TRAIN** (2 months) | each configuration fits ONE model for all the stocks on these stock-days; the last 5 sessions are held out to stop training early, and to set the first month's scale |
| **VALIDATION** (3 months) | the paths are graded; each month's scale β is fit on the held-out week plus the months before it, never on the month itself |
| **TEST** (1 month) | read **once**, for one configuration fixed by a rule before the grid ran (§11), refit on TRAIN + VALIDATION except their last 20 sessions, which give its scale |

Half-days (closed at 13:00) and the session right after each are left out everywhere: a half-day's
afternoon does not exist, and the next morning's input would read a 13:00 close as yesterday's end.

**The guards** (their results join the checks):

1. **The stock-day series, as Darts reads them**: the previous session's last bars plus one whole session,
   the "today" flag exactly on today's bars, never two sessions back.
2. **No night inside any target**: every session's prices are multiplied by a different random factor,
   which changes every overnight gap and nothing within a day. Every target, graded move, unit and
   bar-by-bar input must stay the same, and the gap input must move.
3. **The boundaries as built** (§5): training ends before the held-out week, which ends before the first
   graded decision; every graded path starts at a decision time and ends by 16:00.
4. **The truncation test** (§5): every 1-minute bar after a cut inside VALIDATION is replaced by random
   bars and the whole configuration re-run; every path predicted before the cut, by the network and by
   both baselines, and the first month's scale must be identical to the last digit, and the later paths
   must change (else the test could not fail). It runs on two bar sizes.

In [ ]:
display(ex.split_table(SP))
display(ix.excluded_table(SP, ix.prepared(panel)["m"]["dates"]))
charts.explorer_split(SP, [(SP["validation"][0], "fit (TRAIN)"),
                           (SP["test"][0], "refit (TRAIN + VALIDATION)")]).show()
CHECKS += ix.series_checks(panel, SP, K["default"])
CHECKS += ix.gap_invariance(panel, g0)
display(rs.checks_table(CHECKS[-7:]))

## 5 · One configuration end to end

**Question: for one configuration, what does the model predict, and how close does the path come?**

The worked example (`DEFAULT`): 5-minute bars, input 120 minutes (24 bars), predict 60 minutes (12 bars).
One TiDE model is fit for all the stocks on TRAIN's stock-days, and every decision in VALIDATION gets a
predicted path.

In [ ]:
r0 = ix.run(panel, SP, K["default"], kind=K["model"], epochs=K["epochs"], draws=K["draws"])
CHECKS += ix.run_checks(r0, SP, panel)
TRUNCATE = [K["default"]] if K["smoke"] else [K["default"], (1, 30, 10)]  # the coarsest + finest
for cell in TRUNCATE:
    CHECKS += ix.truncation(panel, SP, cell, kind=K["model"], epochs=K["epochs"])
display(rs.checks_table(CHECKS[-(3 + 2 * len(TRUNCATE)):]))
print(r0["fit"])

**The figure.** The worked decision: the actual price (black, 1-minute closes) and the model's predicted
price at every checkpoint (solid: the graded, scaled path; dotted: the network's own, before scaling).

In [ ]:
charts.path_picture(panel, ix.example_paths(panel, [(f"{g0}-min, input {L0}, predict {S0}", r0)],
                                            NAME, TS), NAME, TS).show()

**The grade.** The truth is the same for every configuration: the log price change from the **fill** (the
first trade after the decision) to each **checkpoint**, in typical moves for that stock, time of day and
length. Each configuration's predicted bar moves are turned into the same thing (a checkpoint between two of
its bar closes, like 10 minutes on 3-minute bars, is read on the straight line between them). Per stock and
checkpoint, **R² = 1 − Σ (predicted − actual)² / Σ actual²**: predicting "no move" scores exactly 0, a path
that helps above 0, one that hurts below. The grade is the mean over the checkpoints, then over the stocks.

**Why the paths are scaled (β).** Most of every move is unpredictable, and a network trained on it still
draws paths too loud for what it knows. Each month's paths are multiplied by β (0 … 1): the factor that
fit best on what was known before that month (the held-out week, then also the earlier graded months).
β = 0 turns a configuration into "no change". The unscaled paths are graded too.

**The baselines,** graded on the same decisions and checkpoints with the same scaling: the **time-of-day
path** (the average path from that decision time over every stock and the previous 20 sessions: the
market's daily pattern) and a **ridge regression** on a dozen summaries known at the decision (the stock's
own last 1 to 60 minutes and since the open, its distance to VWAP, the other stocks' last minutes, volume,
a no-trade flag, the gap, yesterday, the time and the stock), its penalty chosen by cross-validation over
whole days.

**Direction and luck.** Direction is the correlation of predicted and actual change, per stock and
checkpoint, averaged. Its luck band grades the same paths against make-believe futures in which every
10-minute block keeps its size but gets a random sign (one coin per block of the clock, shared by every
stock and decision): direction is gone, everything else stays; z = luck-SDs above luck's centre, which is
exactly 0.

In [ ]:
row0 = pd.to_numeric(pd.Series(ix._row(r0)).drop(["r2_cp", "corr_cp", "ridge_cp", "seasonal_cp",
                                                    "r2_time"]))
display(row0.to_frame("worked example").T.round(4))
display(r0["score"]["r2_by_name"].rename("R² against no change (scaled)").to_frame().T.round(4))
oc0, b0 = ix.prepared(panel)["oc"], ix.baselines(panel, SP, S0)
act0 = oc0["z"][r0["rows"]]
lo, hi = ix.bootstrap(r0["pred"], act0, oc0["s"][r0["rows"]], oc0["k"][r0["rows"]])
lo2, hi2 = ix.bootstrap(r0["pred"], act0, oc0["s"][r0["rows"]], oc0["k"][r0["rows"]],
                        other=b0["ridge"]["pred"])
print(f"R² {r0['score']['r2']:+.4f}, 90% interval {lo:+.4f} … {hi:+.4f}; its lead over the ridge "
      f"{r0['score']['r2'] - b0['ridge']['score']['r2']:+.4f} ({lo2:+.4f} … {hi2:+.4f}); the "
      f"smallest R² these paths could have shown: {ix.mde(r0['score_raw']['null_sd']):.4f}")

**The figures: agreement by checkpoint and by decision time.** R² against "no change" at every checkpoint
of the path (the two baselines in black), then for each decision time on its own.

In [ ]:
one = pd.DataFrame([ix._row(r0)])
charts.checkpoint_curves(one, title="The worked example: R² by checkpoint (VALIDATION)").show()
charts.time_curves(one).show()

## 6 · Which Darts model?

**Question: on the same stocks, inputs and configuration, how do the multi-step Darts models compare?**

In [ ]:
display(ix.models_table(K["compare"]))
comp = ix.compare(panel, SP, K["compare"], K["default"], epochs=K["epochs"], draws=K["draws"])
display(comp[["r2", "beta", "r2_raw", "corr", "z", "ridge_r2", "seasonal_r2", "epochs", "fit_s"]]
        .round(4))
charts.dots(ex.as_dots(comp.reset_index(), "model"), "corr", scale=1.0, unit="",
            title="Direction by model against luck (VALIDATION)").show()

**Look for:** a model whose dot sits outside its grey luck band and whose scaled R² is above 0 and above
the ridge's. Differences inside the bands are not evidence: choose `MODEL` in §0 for what you want to learn.

## 7 · Could this pipeline find an effect if there were one?

**Question: if the data held a real intraday pattern, would this pipeline (stock-day series, fit, scaling,
grade) find it with 12 stocks and these months?**

A known effect is **planted** into the real prices, and the whole worked example is re-run on them: every
minute, each stock drifts in the direction of its own last 30 minutes (a 30-minute momentum, the kind of
pattern only the bar-by-bar inputs reveal). Its strength is set so that the part of the next 30 minutes'
move the past already fixes correlates 0.05, 0.1 or 0.2 with the actual move (R² about 0.0025, 0.01 and
0.04 at 30 minutes; it fades after that). The **oracle** knows the planted rule exactly: its R² is the most
there is to find. If the network finds the larger plants, a "nothing" on the real prices means "nothing of
that size"; if it misses them too, a "nothing" says little.

In [ ]:
plant_rows = []
for rho in K["planted"]:
    pl, info = ix.planted(panel, SP, rho)
    rp = ix.run(pl, SP, K["default"], kind=K["model"], epochs=K["epochs"], draws=K["draws"])
    bp = ix.baselines(pl, SP, S0)
    plant_rows.append({"planted correlation": rho, "oracle R²": ix.oracle(pl, SP, info, S0),
                       "network R²": rp["score"]["r2"], "network z": rp["score_raw"]["z"],
                       "network scale β": ix._row(rp)["beta"],
                       "ridge R²": bp["ridge"]["score"]["r2"]})
plant = ix.planted_table(plant_rows)
display(plant.round(4))
charts.planted_bars(plant).show()
big = plant.iloc[-1]
CHECKS += [("the pipeline finds the largest planted effect (R² > 0, direction z >= 1.96)",
            bool(big["network R²"] > 0 and big["network z"] >= 1.96),
            f"planted corr {plant.index[-1]}: oracle R² {big['oracle R²']:+.4f}, network "
            f"{big['network R²']:+.4f} (z {big['network z']:+.1f})")]

## 8 · The grid: bar size × input span × output span

**Question: which configuration's predicted paths agree best with the actual ones on VALIDATION?**

**The grid.** Every combination of the three knobs is measured (150 cells): small enough to measure all of
it, which gives every knob's effect on balanced data and an honest bar for the best of all the tries. It
runs through Optuna's grid sampler in a random order, so a run cut short is still a fair sample. TEST is not
read until §11.

**What finer bars add.** The table counts, per bar size, the bars each span is, and the **independent**
training paths per stock each output span holds: the same at every bar size. Every bar size also trains on
the same windows (one about every 10 minutes of the clock), so a finer bar adds detail inside each input, not
windows or outcomes to learn from.

In [ ]:
display(ix.grid_table(K["granularities"], K["inputs"], K["outputs"],
                      r0["fit"]["sessions"]))

In [ ]:
res, study = ix.search(panel, SP, kind=K["model"], grans=K["granularities"], inputs=K["inputs"],
                       outputs=K["outputs"], epochs=K["epochs"], draws=K["draws"])
K_FAMILY = len(res) + len(K["compare"]) - 1  # every configuration read on VALIDATION
display(ix.results_table(res, K_FAMILY).head(15))

## 9 · How the three knobs change the picture

**The predicted paths themselves.** The worked decision under the best configuration of each bar size
(solid: graded, scaled; dotted: the network's own).

In [ ]:
best_g = ix.best_per(res, "granularity")  # the best of each bar size (ties: direction z)
runs = [(f"{int(r.granularity)}-min, input {int(r.input)}, predict {int(r.prediction)}",
         ix.run(panel, SP, (r.granularity, r.input, r.prediction), kind=K["model"],
                epochs=K["epochs"], draws=K["draws"])) for r in best_g.itertuples()]
charts.path_picture(panel, ix.example_paths(panel, runs, NAME, TS), NAME, TS).show()

**Agreement by checkpoint.** Every configuration (grey), the best of each bar size (colour) and the best
configuration's two baselines (black): R² against "no change", then the direction, at every checkpoint.
The clearest answer to "how far ahead is the path predictable, and at which bar size?"

In [ ]:
charts.checkpoint_curves(res).show()
charts.checkpoint_curves(res, metric="corr_cp", title="Direction by checkpoint: the best "
                         "configuration of each bar size (VALIDATION)").show()

**Each knob on its own.** One panel per knob: every configuration (grey dots), the best (solid) and the
average (dotted) of each step; first the grade, then the direction's z against luck (grey = where one
configuration lands 95% of the time by luck, dashed = the bar the best of a step's tries must clear).

In [ ]:
PARAMS = {"granularity": ("bar size (minutes)", K["granularities"]),
          "input": ("input span (minutes)", K["inputs"]),
          "prediction": ("output span (minutes)", K["outputs"])}
charts.param_effects(res, PARAMS).show()
charts.param_effects(res, PARAMS, metric="z").show()

**The maps.** One small map per bar size: input span up, output span across, coloured by the grade on one
shared scale (blue above 0, red below), each cell with its R² and direction z; the best configuration is
framed.

In [ ]:
charts.granularity_maps(res, K["granularities"], K["inputs"], K["outputs"], unit="min").show()

**Which decision times, which knob, which order.** R² by decision time for the best configuration of each
output span (the 360-minute span is the forecast from 10:00 to the close), which knob moves the grade most,
and the grid in the order measured.

In [ ]:
charts.time_curves(res).show()
imp = ix.importance(study)
if len(imp):
    charts.importance_bars(imp).show()
charts.search_history(res, unit="min").show()
print(ix.search_words(res, K_FAMILY))

## 10 · Does the start of the day predict the rest of it?

**Question: your alternative design, "always use the first x minutes of the trading day to predict the
rest of the day": is there anything there?**

As a model it gets one example per stock and day (the first x minutes → the rest), about 440 in TRAIN, far
too few for a network and its grade would rest on one path per stock-day. The question itself needs no
model: the **map** below correlates every part of the day with every later part, over all the stock-days
of TRAIN and VALIDATION (the night = the overnight gap, then the thirteen half-hours, the first from the
open). Each stock's moves are divided by their own spread. The luck band flips the sign of each day's
later part, one coin per day for every stock: every move keeps its size, the day its volatility and the
stocks their co-movement, only the direction link goes (a shuffle of days would also break each day's
shared volatility and narrow the band). z = luck-SDs above luck's centre (0); with 91 cells the best of
them must clear the bar for 91 tries. The table below it is your exact question: the first x minutes
against the rest of the day (x = 0: the night alone against the whole session). The models' own answer is
the 360-minute row of §9 (a forecast from 10:00 to the close).

In [ ]:
at = ix.atlas(panel, SP, draws=K["draws"])
charts.atlas_map(at).show()
display(at["first"].round(3))
print(ix.atlas_words(at))

## 11 · TEST: does the winner hold on a month nobody looked at?

**Question: the grid crowned the best of many configurations on VALIDATION. Is it still good on TEST?**

**The rule, fixed before the grid ran:** TEST reads exactly one configuration, the VALIDATION winner (the
best R², ties broken by direction z), refit on TRAIN + VALIDATION except their last 20 usable sessions:
those give its scale (TEST is one month, so no earlier graded month can add to it, as VALIDATION's later
months get), and their last 5 stop its training. It **holds** if its TEST R² is above 0 and its direction
z is at least 1.96. The line also says the smallest R² TEST could have detected: one month is short, so
"did not hold" can mean "could not tell". January 2026 was already read by the explorer's two versions:
this is its third read.

In [ ]:
win = ix.best(res).iloc[0]
cell_w = (int(win["granularity"]), int(win["input"]), int(win["prediction"]))
if READ_TEST:
    rt = ix.run(panel, SP, cell_w, kind=K["model"], epochs=K["epochs"], draws=K["draws"],
                part="test")
    rv = ix.run(panel, SP, cell_w, kind=K["model"], epochs=K["epochs"], draws=K["draws"])
    tv = pd.DataFrame([{"part": "VALIDATION", **ix._row(rv)}, {"part": "TEST", **ix._row(rt)}])
    display(tv.set_index("part")[["granularity", "input", "prediction", "r2", "beta", "corr", "z",
                                  "ridge_r2", "seasonal_r2", "paths"]].round(4))
    charts.checkpoint_curves(tv.assign(label=tv["part"]), by="label",
                             title="The winner: R² by checkpoint, VALIDATION and TEST").show()
    print(ix.test_words(rt, "winner"))
else:
    rt = None
    print("TEST not read (READ_TEST = False)")

## 12 · The checks, the summary, what to try next

In [ ]:
table = rs.checks_table(CHECKS)
display(table)
t_row = ix._row(rt) if rt is not None else {}
rs.scorecard({"notebook": "intraday", "smoke": K["smoke"], "model": K["model"], "names": NAMES,
              "window": SP["window"], "configurations": len(res), "family": K_FAMILY,
              "winner": {"granularity": win["granularity"], "input": win["input"],
                         "prediction": win["prediction"], "val_r2": win["r2"], "val_z": win["z"],
                         "beta": win["beta"], "ridge_r2": win["ridge_r2"],
                         "test_r2": t_row.get("r2"), "test_z": t_row.get("z")},
              "above_no_change": int((res["r2"] > 0).sum()),
              "above_both_baselines": int(((res["r2"] > 0) & (res["vs_best_baseline"] > 0))
                                          .sum()),
              "direction_above_luck": int((res["z"] >= 1.96).sum()),
              "planted": plant.round(4).reset_index().to_dict("records"),
              "checks_ok": f"{int(table['ok'].sum())}/{len(table)}"})

**Reading the result.** §9 and §11 say it in words. Three outcomes are possible:

- **A configuration beats "no change" and the ridge, and holds on TEST**: that (bar size, input span,
  output span) is the intraday baseline to build on.
- **A winner that does not hold**: the usual face of luck after many tries.
- **Nothing beats "no change"**: read it with §7. If the planted effects were found, these stocks' paths
  hold nothing of that size that a model of this kind can learn from two months; if they were missed too,
  the study is too small to say.

**What to try next, one knob at a time:**

| next knob | how | why |
|---|---|---|
| more stocks | `SYMBOLS`, or `ex.PLAN` (more per group) | the smallest detectable R² shrinks with more stocks; about 270 pass the data rules |
| more months | `WINDOW` starting earlier (1-minute data begins 2025-06-18) | more independent days to learn from and to grade |
| a market-relative target | each stock's path minus the average of all | removes the market, which no single stock's history predicts |
| the model | `MODEL` (§6) | TSMixer and DLinear read the inputs differently from TiDE |
| the last 6 months | `WINDOW = ("2026-04-01", "2026-09-30")`, `READ_LOCKED = True` | about 1,750 stocks with 1-minute data, at the price of the locked months |